In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab05.ipynb")

# Lab05 — Statistical Hypothesis Testing and Simulation
**DATA2201 · Foundations of Data Science · Fall 2026**  
**100 points · Fully auto-graded · 7 questions**

**Learning goals:** Analyze a real birth-weight dataset; compare an observed categorical distribution with a *specified teaching null model* using TVD; simulate a null distribution and calculate a p-value; and conduct a guided permutation test of low-birth-weight proportions. These are two different research questions. No statistical test establishes causation.

## Setup — Import libraries

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

## Dataset – Birth Weights

In this lab, we will use the `babyweights.csv` dataset, located in `data/babyweights.csv`, to explore **hypothesis testing and permutation testing** using birth-weight data.

The dataset contains one row per mother–baby pair. We will focus on the following two columns:

| Column | Description | Data Type |
|---|---|---|
| `Birth Weight` | Birth weight of the baby, measured in ounces. | Numeric |
| `Maternal Smoker` | Indicates whether the mother smoked during pregnancy (`True` for smokers and `False` for non-smokers). | Boolean |

### Learning Objectives

Using this dataset, you will:

1. Calculate observed proportions for different birth-weight categories.
2. Compare observed proportions with a specified null model using Total Variation Distance (TVD).
3. Simulate samples under the null model and calculate a simulation-based p-value.
4. Compare the proportions of low-birth-weight babies between smokers and non-smokers.
5. Perform permutation testing to investigate whether the observed difference in proportions is statistically significant.

### Birth-Weight Categories

For this lab, birth weights are divided into three categories:

| Category | Birth Weight (ounces) |
|---|---|
| Below 100 | Less than 100 |
| 100–129 | Greater than or equal to 100 and less than 130 |
| 130 or above | Greater than or equal to 130 |

**Important:** For this lab only, we use **100 ounces as an illustrative classroom threshold** for low birth weight. This threshold is used for educational purposes and does not represent a clinical definition of low birth weight.

### Data Preparation

Run the following code cell to load and prepare the dataset before starting the questions.

The code will:

- Load the dataset into a pandas DataFrame named `baby`.
- Select the two required columns and remove rows containing missing values.
- Convert `Birth Weight` to a numeric data type.
- Convert `Maternal Smoker` to a Boolean data type.
- Verify that the dataset is not empty and contains both smoking-status groups.
- Display the first five rows of the prepared dataset.

**Note:** Use the prepared `baby` DataFrame for all subsequent questions. Do not modify the original data unless instructed.

In [ ]:
baby = pd.read_csv(Path('data') / 'babyweights.csv')
baby = baby[['Birth Weight', 'Maternal Smoker']].dropna().copy()
baby['Birth Weight'] = pd.to_numeric(baby['Birth Weight'], errors='raise')
baby['Maternal Smoker'] = baby['Maternal Smoker'].astype(bool)
assert len(baby) > 0 and baby['Maternal Smoker'].nunique() == 2
baby.head()

## Part A — Hypothesis testing with TVD (50 points)

**Research question:** Does the observed distribution of birth-weight categories differ from the distribution specified by a classroom null model?

- **Null hypothesis (H₀):** The category probabilities are `[0.25, 0.50, 0.25]` for weights below 100 oz, 100–129 oz, and at least 130 oz, respectively.
- **Alternative hypothesis (H₁):** The category distribution differs from this specified model.

The probabilities are a **hypothetical teaching model**, not a published estimate of the population. We use the TVD statistic, `0.5 * sum(abs(observed - null))`. The simulated p-value is the proportion of null-model TVDs at least as large as the observed TVD.

**Worked example — category proportions.** For counts `[2, 5, 3]`, divide by the total `10` to obtain `[0.2, 0.5, 0.3]`.

In [ ]:
example_counts = np.array([2, 5, 3])
example_counts / example_counts.sum()

### Example 1– Counting Birth-Weight Categories

Suppose five babies have birth weights of 95, 110, 125, 135, and 140 ounces.

Using the three birth-weight categories:

- Below 100 ounces: 1 baby
- 100–129 ounces: 2 babies
- 130 ounces or above: 2 babies

The category counts are `[1, 2, 2]`.

Dividing each count by the total number of babies (5) gives the observed proportions:

$[1/5,\ 2/5,\ 2/5] = [0.2,\ 0.4,\ 0.4]$

Use the same approach to calculate the category counts and observed proportions for the actual dataset.

### Question 1 – Birth-Weight Categories and Observed Proportions (15 points)

Using the `Birth Weight` column in the `baby` DataFrame, complete the following tasks:

1. Create a NumPy array named `birth_category_counts` containing the number of babies in each of the following birth-weight categories, in the specified order:
   - **Below 100:** Birth weight less than 100.
   - **100–129:** Birth weight greater than or equal to 100 but less than 130 (`100 <= weight < 130`).
   - **130 or above:** Birth weight greater than or equal to 130.

2. Create a NumPy array named `birth_category_props` containing the observed proportion of babies in each category by dividing the corresponding category counts by the total number of babies.

**Note:** Both arrays should contain three elements, corresponding to the birth-weight categories in the order specified above.

In [ ]:

weights = baby['Birth Weight'].to_numpy()
birth_category_counts = ...
birth_category_props = ...

In [ ]:
grader.check("q1")

**Worked example — TVD.** For `[0.2, 0.5, 0.3]` versus `[0.25, 0.5, 0.25]`, the TVD is `0.05`.

In [ ]:
np.abs(np.array([0.2, 0.5, 0.3]) - np.array([0.25, 0.5, 0.25])).sum() / 2

### Example 2– Comparing Birth-Weight Distributions

Suppose the observed birth-weight category proportions are:

$[0.2,\ 0.4,\ 0.4]$

A hypothetical model specifies the probabilities:

$[0.3,\ 0.4,\ 0.3]$

The Total Variation Distance (TVD) is:

$\text{TVD} = \frac{|0.2-0.3|+|0.4-0.4|+|0.4-0.3|}{2} = 0.1$

A larger TVD indicates a greater difference between the observed distribution and the specified model.

### Question 2 – Observed TVD Under the Null Model (15 points)

Suppose the null model assumes the following probabilities for the three birth-weight categories:

- **Below 100:** 0.25
- **100–129:** 0.50
- **130 or above:** 0.25

Complete the following tasks:

1. Create a NumPy array named `null_probs` containing the three probabilities in the order specified above.
2. Calculate the observed Total Variation Distance (TVD) between `birth_category_props` from Question 1 and `null_probs`. Store the result in `observed_tvd`.

**Formula:**

$
\text{TVD} = \frac{1}{2}\sum_{i=1}^{3}|p_i-q_i|
$

where $p_i$ and $q_i$ represent the observed and null-model probabilities for category \(i\), respectively.

In [ ]:

null_probs = ...
observed_tvd = ...

In [ ]:
grader.check("q2")

**Worked example — multinomial simulation.** `rng.multinomial(n, probs, size=k)` simulates `k` sets of category counts for samples of size `n`. Divide by `n` to obtain proportions.

In [ ]:
example_rng = np.random.default_rng(4)
example_rng.multinomial(10, [0.25, 0.50, 0.25], size=3) / 10

### Example 3– Simulating Birth-Weight Categories

Suppose a hypothetical model specifies birth-weight category probabilities of:

$[0.3,\ 0.4,\ 0.3]$

We simulate a sample of 10 babies under this model.

Suppose one simulated sample produces category counts of:

$[2,\ 5,\ 3]$

The simulated proportions are:

$[0.2,\ 0.5,\ 0.3]$

The TVD for this simulated sample is:

$\text{TVD} = \frac{|0.2-0.3|+|0.5-0.4|+|0.3-0.3|}{2} = 0.1$

Repeating the simulation produces a distribution of TVD values.

The simulated p-value is the proportion of simulated TVDs greater than or equal to the observed TVD.

### Question 3 – Simulated TVD Distribution and p-value (20 points)

Using the null model from Question 2, perform a simulation-based hypothesis test by completing the following tasks:

1. Initialize a random number generator using `np.random.default_rng(41)`.
2. Generate **300 categorical samples**, each containing `len(baby)` observations, using the probabilities in `null_probs`. Store the simulated sample proportions in a NumPy array named `simulated_props` with shape `(300, 3)`.
3. Calculate the TVD between each simulated sample's proportions and `null_probs`. Store the 300 TVD values in `null_tvds`.
4. Calculate the **right-tail p-value** as the proportion of simulated TVDs greater than or equal to `observed_tvd`. Store the result in `tvd_p_value`.

**Note:** Use the TVD formula from Question 2. The p-value should be a single numerical value between 0 and 1.

In [ ]:

tvd_rng = np.random.default_rng(41)
simulated_props = ...
null_tvds = ...
tvd_p_value = ...

In [ ]:
grader.check("q3")

## Part B — Permutation testing of low-birth-weight proportions (50 points)

**Research question:** Does the proportion of babies weighing below 100 ounces differ between the maternal-smoking groups in this dataset?

- **Null hypothesis (H₀):** The low-birth-weight outcome and maternal-smoking label are unrelated; the observed labels are exchangeable.
- **Alternative hypothesis (H₁):** The proportions differ between the two groups.

The statistic is the **absolute difference in proportions**, so use a **two-sided** permutation p-value. Shuffle the low-weight outcomes while keeping the smoking labels fixed. This is a comparison of associations in observational data, **not a causal claim**.

**Worked example — difference in proportions.** If 2 of 5 observations in group A and 1 of 5 in group B meet a condition, the absolute difference is `abs(2/5 - 1/5) = 0.2`.

In [ ]:
abs(2 / 5 - 1 / 5)

### Example 4– Comparing Smokers and Non-Smokers

Suppose we observe the following birth-weight outcomes:

| Maternal Smoking Status | Total Babies | Babies Below 100 oz |
|---|---:|---:|
| Smoker | 10 | 4 |
| Non-smoker | 10 | 2 |

The proportion of low-birth-weight babies among smokers is:

$4/10 = 0.4$

The proportion among non-smokers is:

$2/10 = 0.2$

The absolute difference in proportions is:

$|0.4-0.2| = 0.2$

Use the actual birth-weight data to calculate the observed difference between the two groups.

### Question 4 – Observed Difference in Low-Birth-Weight Proportions (15 points)

Using the `baby` DataFrame, complete the following tasks:

1. Create a Boolean NumPy array named `low_weight` using the `Birth Weight` column. Each element should be `True` if the baby's birth weight is below 100 ounces and `False` otherwise.

2. Create a Boolean NumPy array named `smoker_labels` using the `Maternal Smoker` column (`True` for smokers and `False` for non-smokers).

3. Calculate the proportion of low-birth-weight babies separately for smokers and non-smokers. Store the **absolute difference** between these two proportions in `observed_prop_diff`.

**Note:** The observed difference should be a single non-negative numerical value.

In [ ]:

low_weight = ...
smoker_labels = ...
observed_prop_diff = ...

In [ ]:
grader.check("q4")

**Worked example — a single shuffle.** A permutation changes the order but preserves the number of `True` and `False` outcomes.

In [ ]:
np.random.default_rng(8).permutation(np.array([True, False, True, False]))

### Example 5– Shuffling Birth-Weight Outcomes

Suppose we have six babies with the following data:

| Baby | Maternal Smoker | Below 100 oz |
|---|---|---|
| 1 | True | True |
| 2 | True | True |
| 3 | True | False |
| 4 | False | False |
| 5 | False | False |
| 6 | False | True |

The original difference in low-birth-weight proportions is:

$|2/3-1/3| = 1/3$

Now suppose we randomly shuffle the low-birth-weight outcomes while keeping the smoking labels unchanged.

One possible shuffled outcome is:

`[False, True, False, True, False, True]`

The difference after shuffling is:

$|1/3-2/3| = 1/3$

This is one possible permutation test statistic. Different shuffles may produce different values.

### Question 5 – One Permutation and Its Test Statistic (15 points)

Using the `low_weight` and `smoker_labels` arrays from Question 4, complete the following tasks:

1. Initialize a random number generator using `np.random.default_rng(52)`.
2. Randomly shuffle `low_weight` once using the generator's `permutation()` method. Store the shuffled outcomes in `shuffled_low`.
3. Calculate the proportion of low-birth-weight babies separately for smokers and non-smokers using `shuffled_low` and the **original, unchanged** `smoker_labels`.
4. Store the absolute difference between these two proportions in `one_perm_diff`.

**Note:** Shuffle only the low-birth-weight outcomes, not the smoking labels. The test statistic should be a single non-negative numerical value.

In [ ]:

perm_rng = np.random.default_rng(52)
shuffled_low = ...
one_perm_diff = ...

In [ ]:
grader.check("q5")

### Example 6– Repeated Permutations

Instead of shuffling the birth-weight outcomes only once, we repeat the permutation procedure multiple times.

Suppose five permutations produce the following absolute differences in low-birth-weight proportions:

$[0.10,\ 0.20,\ 0.00,\ 0.30,\ 0.10]$

These values form a small permutation distribution.

Each value represents the difference between smokers and non-smokers after randomly shuffling the low-birth-weight outcomes while keeping the original smoking labels unchanged.

Repeating the process many times allows us to examine how much the difference in proportions varies under the null hypothesis.

### Question 6 – Permutation Distribution (10 points)

Extend the permutation procedure from Question 5 to generate a distribution of simulated test statistics.

1. Initialize a new random number generator using `np.random.default_rng(53)`.
2. Repeat the permutation procedure **200 times**, shuffling `low_weight` while keeping the original `smoker_labels` unchanged.
3. After each shuffle, calculate the absolute difference in low-birth-weight proportions between smokers and non-smokers.
4. Store the 200 simulated differences in a NumPy array named `perm_diffs`.

**Note:** Use a loop to perform the 200 permutations.

In [ ]:

perm_rng2 = np.random.default_rng(53)
perm_diffs = []
for _ in range(200):
    shuffled = ...
    perm_diffs.append(...)
perm_diffs = np.asarray(perm_diffs)

In [ ]:
grader.check("q6")

### Example 7– Interpreting the Permutation p-value

Suppose the observed absolute difference in low-birth-weight proportions between smokers and non-smokers is:

$0.30$

After performing 100 permutations, suppose 4 simulated differences are greater than or equal to 0.30.

The permutation p-value is:

$\text{p-value} = \frac{4}{100} = 0.04$

At a significance level of 0.05:

$0.04 < 0.05$

Therefore, we reject the null hypothesis.

This provides statistical evidence of an association between maternal smoking status and low-birth-weight outcomes in this hypothetical example. It does not establish that smoking caused the observed difference.

### Question 7 – Permutation p-value and Statistical Decision (10 points)

Using the permutation distribution from Question 6, complete the following tasks:

1. Calculate the **right-tail p-value** as the proportion of values in `perm_diffs` that are greater than or equal to `observed_prop_diff`. Store the result in `perm_p_value`.

2. Set `reject_perm_null` to `True` if the p-value is strictly less than 0.05, indicating rejection of the null hypothesis. Otherwise, set it to `False`.

**Note:** The p-value should be a single numerical value between 0 and 1, and `reject_perm_null` should be a Boolean value.

In [ ]:

perm_p_value = ...
reject_perm_null = ...

In [ ]:
grader.check("q7")

## Congratulations! You have finished Lab 05!


Congrats! You are finished with this assignment.

## Submission

Run **Kernel → Restart & Run All** and verify that all public tests pass. Submit the `.ipynb` file according to the course instructions. The hidden tests are evaluated by the autograder. The null-model probabilities and 100-ounce threshold are classroom assumptions; neither is a clinical or population benchmark.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)